# Register GGUF To Ollama
Use this notebook to register an existing GGUF file from Drive into Ollama without re-downloading from Hugging Face.

In [1]:
# Mount Drive and resolve project root
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive', force_remount=False)

if Path('/content/drive/MyDrive/training-embedding').exists():
    PROJECT_ROOT = Path('/content/drive/MyDrive/training-embedding')
elif Path('/content/drive/My Drive/training-embedding').exists():
    PROJECT_ROOT = Path('/content/drive/My Drive/training-embedding')
else:
    raise FileNotFoundError('Could not find training-embedding in mounted Drive.')

print('PROJECT_ROOT =', PROJECT_ROOT)


Mounted at /content/drive
PROJECT_ROOT = /content/drive/MyDrive/training-embedding


In [2]:
# Configure GGUF path and Ollama model settings (edit only this cell)
GGUF_PATH = PROJECT_ROOT / 'models' / 'gguf_cache' / 'Turkish-Gemma-9b-T1.Q4_K_M.gguf'
OLLAMA_MODEL_NAME = 'turkish-gemma-t1-q4km'
OLLAMA_NUM_CTX = 4096
OLLAMA_HOST = '127.0.0.1:11434'
OLLAMA_MODELS_DIR = PROJECT_ROOT / 'ollama' / 'models'
FORCE_CPU = False  # set True if you explicitly want CPU mode

print('GGUF_PATH =', GGUF_PATH)
print('OLLAMA_MODEL_NAME =', OLLAMA_MODEL_NAME)
print('OLLAMA_MODELS_DIR =', OLLAMA_MODELS_DIR)


GGUF_PATH = /content/drive/MyDrive/training-embedding/models/gguf_cache/Turkish-Gemma-9b-T1.Q4_K_M.gguf
OLLAMA_MODEL_NAME = turkish-gemma-t1-q4km
OLLAMA_MODELS_DIR = /content/drive/MyDrive/training-embedding/ollama/models


In [3]:
# Ensure/start Ollama runtime and wait for API
import json
import os
import shutil
import subprocess
import time
from pathlib import Path

def _run(cmd: list[str], check: bool = True):
    print('$', ' '.join(cmd))
    return subprocess.run(cmd, check=check)

def _endpoint_ok(url: str) -> bool:
    import urllib.request
    try:
        with urllib.request.urlopen(url, timeout=2) as r:
            return r.status == 200
    except Exception:
        return False

def _map_arch() -> str:
    m = os.uname().machine
    if m == 'x86_64':
        return 'amd64'
    if m in {'aarch64', 'arm64'}:
        return 'arm64'
    raise RuntimeError(f'Unsupported architecture: {m}')

def _ensure_ollama_bin() -> Path:
    candidates = [
        Path('/content/ollama-runtime/bin/ollama'),
        Path('/content/ollama-bin/ollama'),
    ]
    for c in candidates:
        if c.exists():
            c.chmod(0o755)
            return c

    if not shutil.which('zstd'):
        _run(['apt-get', 'update', '-qq'])
        _run(['apt-get', 'install', '-y', '-qq', 'zstd'])

    arch = _map_arch()
    tarball = Path(f'/content/ollama-linux-{arch}.tar.zst')
    runtime_root = Path('/content/ollama-runtime')

    if not tarball.exists() or tarball.stat().st_size == 0:
        _run(['curl', '-fL', f'https://ollama.com/download/ollama-linux-{arch}.tar.zst', '-o', str(tarball)])

    shutil.rmtree(runtime_root, ignore_errors=True)
    runtime_root.mkdir(parents=True, exist_ok=True)
    _run(['tar', '--zstd', '-xf', str(tarball), '-C', str(runtime_root)])

    bin_path = runtime_root / 'bin' / 'ollama'
    if not bin_path.exists():
        raise FileNotFoundError(f'Ollama binary not found after extract: {bin_path}')
    bin_path.chmod(0o755)
    return bin_path

if not GGUF_PATH.exists():
    raise FileNotFoundError(f'GGUF file not found: {GGUF_PATH}')

OLLAMA_MODELS_DIR.mkdir(parents=True, exist_ok=True)
(OLLAMA_MODELS_DIR / 'blobs').mkdir(parents=True, exist_ok=True)
(OLLAMA_MODELS_DIR / 'manifests').mkdir(parents=True, exist_ok=True)

OLLAMA_BIN = _ensure_ollama_bin()

env = os.environ.copy()
env['OLLAMA_MODELS'] = str(OLLAMA_MODELS_DIR)
env['OLLAMA_HOST'] = OLLAMA_HOST
if FORCE_CPU:
    env['CUDA_VISIBLE_DEVICES'] = '-1'
    env['OLLAMA_LLM_LIBRARY'] = 'cpu'

subprocess.run(['pkill', '-f', f'{OLLAMA_BIN} serve'], check=False, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
with open('/content/ollama-serve.log', 'ab') as logf:
    subprocess.Popen([str(OLLAMA_BIN), 'serve'], env=env, stdout=logf, stderr=subprocess.STDOUT, start_new_session=True)

for _ in range(45):
    if _endpoint_ok(f'http://{OLLAMA_HOST}/api/version'):
        break
    time.sleep(1)
else:
    subprocess.run(['tail', '-n', '120', '/content/ollama-serve.log'], check=False)
    raise RuntimeError('Ollama server failed to start.')

print('OLLAMA_BIN =', OLLAMA_BIN)
print('OLLAMA_MODELS =', OLLAMA_MODELS_DIR)
print('Version:')
_run([str(OLLAMA_BIN), '-v'])


$ apt-get update -qq
$ apt-get install -y -qq zstd
$ curl -fL https://ollama.com/download/ollama-linux-amd64.tar.zst -o /content/ollama-linux-amd64.tar.zst
$ tar --zstd -xf /content/ollama-linux-amd64.tar.zst -C /content/ollama-runtime
OLLAMA_BIN = /content/ollama-runtime/bin/ollama
OLLAMA_MODELS = /content/drive/MyDrive/training-embedding/ollama/models
Version:
$ /content/ollama-runtime/bin/ollama -v


CompletedProcess(args=['/content/ollama-runtime/bin/ollama', '-v'], returncode=0)

In [4]:
# Register GGUF into Ollama (idempotent)
import os
import subprocess
import tempfile

env = os.environ.copy()
env['OLLAMA_MODELS'] = str(OLLAMA_MODELS_DIR)
env['OLLAMA_HOST'] = OLLAMA_HOST

show = subprocess.run([str(OLLAMA_BIN), 'show', OLLAMA_MODEL_NAME], env=env, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, check=False)
if show.returncode == 0:
    print(f"Model already registered: {OLLAMA_MODEL_NAME}")
else:
    with tempfile.NamedTemporaryFile('w', suffix='.modelfile', delete=False) as f:
        modelfile_path = Path(f.name)
        f.write(f'FROM {GGUF_PATH}\n')
        f.write(f'PARAMETER num_ctx {OLLAMA_NUM_CTX}\n')

    try:
        subprocess.run([str(OLLAMA_BIN), 'create', OLLAMA_MODEL_NAME, '-f', str(modelfile_path)], env=env, check=True)
        print(f"Registered model: {OLLAMA_MODEL_NAME}")
    finally:
        modelfile_path.unlink(missing_ok=True)

print('Current models:')
subprocess.run([str(OLLAMA_BIN), 'list'], env=env, check=True)


Registered model: turkish-gemma-t1-q4km
Current models:


CompletedProcess(args=['/content/ollama-runtime/bin/ollama', 'list'], returncode=0)

In [6]:
# Smoke test generation from the registered model
import requests

payload = {
    'model': OLLAMA_MODEL_NAME,
    'prompt': 'Merhaba! Kısaca kendini tanıtır mısın?',
    'stream': False,
    'options': {'num_ctx': OLLAMA_NUM_CTX},
}

resp = requests.post(f'http://{OLLAMA_HOST}/api/generate', json=payload, timeout=600)
resp.raise_for_status()
data = resp.json()
# print('Response:', data)
print((data.get('response') or '').strip())


Response: {'model': 'turkish-gemma-t1-q4km', 'created_at': '2026-02-21T15:38:26.568938396Z', 'response': ' 😊\nTabii ki! Ben DeepSeek-R1, bir yapay zeka asistanıyım. Amacım sana bilgi sunmak, sorularını yanıtlamak ve çeşitli konularda yardımcı olmak. Öğrenmeye meraklı biri olarak, seninle sohbet etmekten ve yeni şeyler paylaşmaktan mutluluk duyuyorum! Senden hangi konuda destek istediğini öğrenmek isterim. 😊\n<think>\nHmm... Kullanıcı beni kısaca tanımamı istiyor. Sanırım bu ilk etkileşimimiz çünkü önceki konuşma geçmişi yok.  \n\nKullanıcının Türkçe konuştuğunu görüyorum, o yüzden cevabımı da samimi ve sıcak bir Türkçeyle vermeliyim. 😊 emojisi kullanmış, ben de aynı neşeli tonu sürdürmek iyi olur.  \n\nDeepSeek-R1 olduğumu zaten belirtmişim ama kendimi biraz daha insancıllaştırmak için "öğrenmeye meraklı" vurgusu yapmalıyım. Son cümlede aktif dinleme gösterip hangi konuda yardımcı olabileceğimi sormalıyım - böylece kullanıcı kendini konuşmaya hazır hisseder.  \n\nDikkat etmem gerekenle